# part별 공정조합 평균 · 순위 정규변환 / Yeo–Johnson / raw

기본 입력은 실제 **`raw.csv`**입니다. **`Y_TRANSFORM = 'rank_normal'`**이 기본이며, `'yeo_johnson'`이면 기존 Yeo–Johnson, `'raw'`이면 변환 OFF입니다. 설정을 바꾼 뒤 **Run All** 하세요. Yeo–Johnson은 `YJ_LAMBDA_METHOD='robust'` 또는 `'mle'`로 선택합니다.

**공정조합 생성 규칙은 기존 raw 버전을 유지합니다.** 웨이퍼 포함 조건, 단일 유닛 Order 제외, 최소 웨이퍼 수, raw y와 G/B에 의한 부모 조합 선별은 변환 옵션과 무관합니다. 동일 입력·설정에서 후보 조합과 웨이퍼 수, $m_{p,n}$이 같습니다. 변환은 후보 선별 후 표시·판정할 평균에만 사용합니다. 원본 Bonferroni 최종 판정으로 점을 거르지 않습니다.

### 순위 정규변환
해당 part의 분석 대상 **고유 웨이퍼** 전체에서 raw y의 순위 $r_i$를 구하고 다음 점수로 바꿉니다. 공정 이력 행을 중복으로 세거나 조합별로 순위를 다시 구하지 않습니다.

$$z_i=\Phi^{-1}\!\left(\frac{r_i-0.5}{K_p}\right),\qquad r_i=\operatorname{average\ rank}(y_i)$$

**동일한 y는 평균 순위와 동일 점수**를 받습니다. 행 순서로 동률을 깨거나 노이즈를 넣지 않습니다. 동률이 없으면 정규 분위수 격자를 얻지만, 정확히 0 등 같은 값이 많으면 그 덩어리는 남아 정규분포가 되지 않습니다. 최대 동률 비율과 변환 전·후 히스토그램/Q–Q plot을 확인하세요. 실제 점수의 평균·표준편차를 계산하며 0·1로 강제하지 않습니다.

큰 y가 더 커져도 순서가 같으면 점수는 같습니다. 타점은 조합 내 **상대적으로 높은 y가 얼마나 모이는지**를 나타내는 평균 정규점수입니다. '과반수 웨이퍼가 불량'이라는 명제 자체를 검정하는 것은 아니며, 한 높은 순위의 영향도 완전히 0이 되지는 않습니다. raw 평균은 함께 보존합니다. 순위변환은 raw 단위의 역변환 경계를 제공하지 않습니다.

### 팀원 원본의 순위점수와 비교
원본 `real_data_funnel.ipynb`도 `-norm_isf((rank - 0.5) / K)`로 순위를 정규점수로 바꿉니다. 이는 위의 역정규누적분포 공식과 같으며, pandas 기본 평균 순위로 동률을 처리한다는 점도 같습니다. 새로운 종류의 순위변환을 도입한 것은 아닙니다.

| 항목 | 팀원 원본 | 이 노트북 |
|---|---|---|
| 순위 기준 집단 | 선택된 모든 part의 웨이퍼를 합산 (`PART_ID=None` 기본) | part별 고유 웨이퍼 각각 |
| 적용 여부 | 기본 `auto`: 절대 왜도가 1을 넘을 때 순위변환 | `rank_normal`로 명시 선택, 현재 기본값 |
| part 평균 보정 | 조건을 만족하면 순위변환 뒤 part 평균을 빼고 전체 평균을 더함 | 추가 평균 보정 없이 part별 분석·차트 |
| 부모 조합 선별 | 변환·보정된 신호로 부모 z를 계산 | 기존 raw 신호를 유지하여 변환 옵션 간 후보가 동일 |
| 최종 envelope | 6절에서 유한모집단 보정, N별 퍼짐 보정 등의 항을 적용 | part별 점수의 표준편차 / √n × √(2 ln m_n) |

원본에서도 **단일 part만 선택하고 순위변환을 켜면**, 동일한 분석 웨이퍼와 같은 값의 방향을 썼을 때 개별 웨이퍼 점수는 사실상 같습니다. 차이는 여러 part 처리와 후속 선별·경계 계산입니다. 원본의 part 평균 이동만으로는 part별 분포 모양·꼬리 차이를 없앨 수 없습니다.

### 평균 타점과 envelope
`rank_normal`이면 위 점수, `yeo_johnson`이면 $T_{\lambda_p}(y)$, `raw`이면 $y$를 $z$로 사용합니다.

$$\hat\mu_p=\frac1{K_p}\sum_i z_{p,i},\qquad
\hat\sigma_p=\sqrt{\frac1{K_p-1}\sum_i(z_{p,i}-\hat\mu_p)^2}$$
$$m_{p,n}=\text{같은 part에서 웨이퍼 수가 정확히 }n\text{인 표시 공정조합 수}$$
$$\text{envelope}_{p,n}=\hat\mu_p\pm\frac{\hat\sigma_p}{\sqrt n}\sqrt{2\ln m_{p,n}}$$

타점은 같은 조합에 속한 웨이퍼의 $z$ 평균입니다. 표준편차는 part당 하나이며 같은 n의 조합 평균들에서 재추정하지 않습니다. $m_{p,n}=1$이면 식 그대로 폭은 0입니다. 추가 표준오차 보정이나 평활화는 하지 않습니다.

이 경계는 동일 기준 분포와 독립 평균 타점을 가정한 **정규 기대 극값의 1차 근사**입니다. 동률이 많은 순위 점수와 Yeo–Johnson 결과에 정규성이 보장되는 것은 아닙니다. 결과에는 원단위 `raw_mean`, `mu_hat_raw`, `sigma_hat_raw`와 변환 방식을 남깁니다.

### Yeo–Johnson 옵션
`robust`는 Raymaekers–Rousseeuw의 [RewML 연구](https://doi.org/10.1007/s10994-021-05960-5)에 기반한 raw 입력용 구현입니다. 강건한 초기 Q–Q 적합과 두 번의 재가중 최대우도 추정으로 part당 λ를 한 번 정합니다. 초기 적합의 보조 변환과 달리 실제 타점에는 일반 Yeo–Johnson을 사용합니다. 입력의 사전 중심 이동·척도 조정, 출력 평균 0·표준편차 1 표준화는 하지 않습니다.

λ 추정 가중치가 0인 웨이퍼도 최종 타점과 전체 평균·표준편차 계산에 포함됩니다. 강건 추정이 불가능하면 기본적으로 MLE로 전환하고 사유를 기록합니다. `YJ_ROBUST_FALLBACK='error'`이면 해당 part를 제외합니다. `mle`는 이전 전체 데이터 최대우도 추정입니다. **순위 정규변환에는 λ가 없습니다.**

CSV·집계·재작업 정리는 Polars, 조합 집계는 정수 키를 사용합니다. 필요 패키지: `polars>=1.0`, `numpy`, `pandas`, `matplotlib`, `ipython`. 변환을 켜면 `scipy`도 필요합니다.
설치 예: `%pip install "polars>=1.0" scipy numpy pandas matplotlib ipython`

저장소 루트에 `raw.csv`를 두고 Run All. 다른 위치는 `CSV_PATH`를 지정합니다. 기존 `real_data_funnel.ipynb`는 수정하지 않습니다.


In [ ]:
from pathlib import Path

# notebooks 폴더에서 실행하면 저장소 루트의 raw.csv를 읽습니다.
DATA_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CSV_PATH = DATA_DIR / 'raw.csv'
# CSV_PATH = Path('/absolute/path/to/raw.csv')

Y_TRANSFORM = 'rank_normal'  # 'rank_normal': 순위 정규변환 / 'yeo_johnson': YJ / 'raw': 변환 OFF
# 변환 옵션을 바꾸면 Run All. YJ_* 설정은 Y_TRANSFORM='yeo_johnson'일 때만 사용합니다.
YJ_LAMBDA_METHOD = 'robust'   # 'robust': RewML 기반 강건 추정 / 'mle': 이전 전체 데이터 MLE
YJ_ROBUST_FALLBACK = 'mle'    # 강건 추정 불가: 'mle'로 명시 전환 / 'error'이면 해당 part 제외

PART_IDS = None              # None: 모든 part. 예: ['PRODUCT_A', 'PRODUCT_B']
LINE_ID = None               # 특정 line만 사용할 때 지정
STEP = None                  # None: 모든 STEP. 예: 'ex102000'
MIN_N = 20                   # 실제 조합의 최소 고유 웨이퍼 수
MAX_DEPTH = 3                # 한 조합에 묶을 Order 수: 1 ~ MAX_DEPTH
STEP_SUFFIX = r'_(\d+)$'     # step_seq 끝의 _숫자를 Order로 해석. None이면 step_ord 사용
REWORK = 'last'              # 같은 wafer·STEP·Order 재작업: 'first' 또는 'last'
LOG_X = True
OUTSIDE_ROWS = 30            # 표에 표시할 경계 밖 조합 수. 차트에는 모든 점 표시
SAVE_DIR = None              # 예: DATA_DIR / 'part_funnel_results'
HIGHER_IS_WORSE = True       # raw 후보 선별 방향. 표시값의 부호는 뒤집지 않음
SHOW_PROGRESS = True        # part 시작과 긴 계산의 진행 상황 표시


In [ ]:
import hashlib
import html
import math
import time
import re
from itertools import combinations

import numpy as np
import pandas as pd
try:
    import polars as pl
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError('Polars가 필요합니다. Jupyter 셀에서 %pip install "polars>=1.0" 실행 후 커널을 재시작하세요.') from exc
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import HTML, display

assert isinstance(MIN_N, int) and not isinstance(MIN_N, bool) and MIN_N >= 1
assert isinstance(MAX_DEPTH, int) and not isinstance(MAX_DEPTH, bool) and MAX_DEPTH >= 1
assert REWORK in ('first', 'last')
assert Y_TRANSFORM in ('rank_normal', 'yeo_johnson', 'raw')
assert YJ_LAMBDA_METHOD in ('robust', 'mle')
assert YJ_ROBUST_FALLBACK in ('mle', 'error')
assert isinstance(OUTSIDE_ROWS, int) and OUTSIDE_ROWS >= 1

font_dirs = [Path.cwd() / 'fonts', Path.cwd() / 'notebooks' / 'fonts']
font_file = next((d / 'NanumGothic-Regular.ttf' for d in font_dirs if (d / 'NanumGothic-Regular.ttf').is_file()), None)
if font_file is not None:
    mpl.font_manager.fontManager.addfont(str(font_file))
    font_name = mpl.font_manager.FontProperties(fname=str(font_file)).get_name()
else:
    available = {f.name for f in mpl.font_manager.fontManager.ttflist}
    font_name = next((name for name in ['NanumGothic', 'Noto Sans CJK KR', 'Malgun Gothic', 'AppleGothic'] if name in available), 'DejaVu Sans')
plt.rcParams.update({'font.family': font_name, 'axes.unicode_minus': False, 'figure.dpi': 120})
pd.set_option('display.max_columns', 20)
pd.set_option('display.precision', 8)

WAFER_KEY = ['root_lot_id', 'wafer_id']
REQUIRED = ['part_id', 'root_lot_id', 'wafer_id', 'y_value', 'step_seq', 'eqp_id']
OPTIONAL = ['chamber_id', 'step_ord', 'tkin_time', 'line_id', 'good_bad']
MISSING = 0xFFFF

def part_heading(part_id):
    display(HTML('<h3>part_id = ' + html.escape(str(part_id)) + '</h3>'))

## 1. 실제 CSV 읽기 — Polars
필요한 열만 읽고, part별 웨이퍼와 공정 이력을 준비합니다. 같은 웨이퍼의 y가 여러 개면 원본처럼 첫 유효 값을 사용하고 건수를 알립니다.
원본과 같이 G/B 값이 존재하면 해당 웨이퍼만 분석합니다. G/B가 전혀 없으면 raw y만으로 분석합니다.


In [ ]:
def read_raw_csv(path):
    path = Path(path).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f'실제 raw.csv를 찾을 수 없습니다: {path}\nCSV_PATH를 지정하세요.')
    columns = pl.read_csv(path, n_rows=0, infer_schema=False).columns
    missing = sorted(set(REQUIRED) - set(columns))
    if missing:
        raise ValueError(f'CSV에 필요한 컬럼이 없습니다: {missing}')
    if LINE_ID is not None and 'line_id' not in columns:
        raise ValueError('LINE_ID를 지정했지만 CSV에 line_id가 없습니다.')
    use = REQUIRED + [c for c in OPTIONAL if c in columns]
    # pandas 기본 결측 표기와 동일하게 읽는다. 식별자의 앞자리 0은 문자열로 보존한다.
    null_values = ['', '#N/A', '#N/A N/A', '#NA', '-1.#IND', '-1.#QNAN', '-NaN', '-nan',
                   '1.#IND', '1.#QNAN', '<NA>', 'N/A', 'NA', 'NULL', 'NaN', 'None', 'n/a', 'nan', 'null']
    raw = pl.read_csv(path, columns=use, infer_schema=False, null_values=null_values)
    rows_read = raw.height
    raw = raw.with_columns([pl.col(c).str.strip_chars().replace('', None).alias(c) for c in use])
    if LINE_ID is not None:
        raw = raw.filter(pl.col('line_id') == str(LINE_ID))
    missing_key = raw.select(pl.any_horizontal(pl.col(['part_id'] + WAFER_KEY).is_null()).sum()).item()
    raw = raw.drop_nulls(['part_id'] + WAFER_KEY)
    found_parts = sorted(raw['part_id'].unique().to_list())
    selected = found_parts if PART_IDS is None else ([str(PART_IDS)] if isinstance(PART_IDS, (str, int)) else list(dict.fromkeys(map(str, PART_IDS))))
    unknown = sorted(set(selected) - set(found_parts))
    if unknown:
        raise ValueError(f'CSV에 없는 part_id: {unknown}')
    if not selected:
        raise ValueError('분석할 part가 없습니다.')
    raw = raw.filter(pl.col('part_id').is_in(selected)).with_row_index('_row')
    wid_number = pl.col('wafer_id').cast(pl.Float64, strict=False)
    raw = raw.with_columns(
        pl.when(wid_number.is_finite() & (wid_number == wid_number.floor()))
          .then(wid_number.cast(pl.Int64, strict=False).cast(pl.String))
          .otherwise(pl.col('wafer_id')).alias('wafer_id'),
        pl.col('y_value').cast(pl.Float64, strict=False).alias('_y'),
    ).with_columns(pl.when(pl.col('_y').is_finite()).then(pl.col('_y')).otherwise(None).alias('_y'))
    if 'chamber_id' not in raw.columns:
        raw = raw.with_columns(pl.lit('').alias('chamber_id'))
    else:
        raw = raw.with_columns(pl.col('chamber_id').fill_null(''))
    if 'good_bad' in raw.columns:
        raw = raw.with_columns(pl.col('good_bad').str.to_uppercase().alias('_gb'))
        raw = raw.with_columns(pl.when(pl.col('_gb') == 'G').then(0.0)
                              .when(pl.col('_gb') == 'B').then(1.0).otherwise(None).alias('_bad'))
    else:
        raw = raw.with_columns(pl.lit(None, dtype=pl.String).alias('_gb'), pl.lit(None, dtype=pl.Float64).alias('_bad'))
    # 시각 해석은 원본과 같은 pandas 규칙을 사용하고, 정렬·집계는 Polars에서 수행한다.
    if 'tkin_time' in raw.columns:
        times = pd.to_datetime(raw['tkin_time'].to_numpy(), errors='coerce', utc=True)
        raw = raw.with_columns(pl.Series('_time', times.to_numpy(dtype='datetime64[ns]')))
    else:
        raw = raw.with_columns(pl.lit(None, dtype=pl.Datetime('ns')).alias('_time'))
    fallback = pl.col('step_ord').cast(pl.Float64, strict=False) if 'step_ord' in raw.columns else pl.lit(None, dtype=pl.Float64)
    if STEP_SUFFIX:
        raw = raw.with_columns(pl.col('step_seq').str.replace_all(STEP_SUFFIX, '').alias('_step'),
                               pl.coalesce(pl.col('step_seq').str.extract(STEP_SUFFIX, 1).cast(pl.Float64, strict=False), fallback).alias('_order'))
    else:
        raw = raw.with_columns(pl.col('step_seq').alias('_step'), fallback.alias('_order'))
    print(f'실제 데이터: {path}\nPolars {pl.__version__} · 읽은 행 {rows_read:,} · 식별자 결측 제외 {missing_key:,} · part {len(selected):,}개')
    return raw, selected


raw, selected_parts = read_raw_csv(CSV_PATH)
inventory = raw.group_by('part_id').len().sort('part_id')
display(pd.DataFrame(inventory.to_dict(as_series=False)).rename(columns={'len': '이력 행 수'}))


## 2. 원본과 같은 공정조합 후보 생성
- 각 STEP에서 유닛이 2개 이상인 Order만 사용합니다.
- 깊이 1~2는 MIN_N을 만족하는 관측 조합을 계산합니다.
- 깊이 3 이상은 원본과 같이 부모 조합의 z 관문을 통과한 후보만 계산합니다.
- STEP 설정은 원본처럼 표시 범위만 정합니다. 후보 생성과 부모 선별은 전체 STEP에서 먼저 수행합니다.

순위 정규변환 / Yeo–Johnson / raw 중 어느 것을 선택해도 부모 선별에는 기존 raw y를 사용하므로 동일 입력의 후보 수가 같습니다.
아래 수치 함수와 후보 생성 함수는 원본 구현에서 가져왔습니다. 작은 정수 키는 정렬 없이 집계하도록 최적화했습니다.


In [ ]:
Z95, FWER_ALPHA = 1.959964, 0.05
SQRT2 = math.sqrt(2)
_A = (-39.69683028665376, 220.9460984245205, -275.9285104469687, 138.357751867269, -30.66479806614716, 2.506628277459239)
_B = (-54.47609879822406, 161.5858368580409, -155.6989798598866, 66.80131188771972, -13.28068155288572)
_C = (-0.007784894002430293, -0.3223964580411365, -2.400758277161838, -2.549732539343734, 4.374664141464968, 2.938163982698783)
_D = (0.007784695709041462, 0.3224671290700398, 2.445134137142996, 3.754408661907416)
_LG = (76.18009172947146, -86.50532032941677, 24.01409824083091, -1.231739572450155, 0.1208650973866179e-2, -0.5395239384953e-5)


def erfc(x):
    x = np.asarray(x, dtype=float)
    z = np.abs(x)
    t = 1 / (1 + 0.5 * z)
    poly = 1.00002368 + t * (0.37409196 + t * (0.09678418 + t * (-0.18628806 + t * (0.27886807 + t * (-1.13520398 + t * (1.48851587 + t * (-0.82215223 + t * 0.17087277)))))))
    r = t * np.exp(-z * z - 1.26551223 + t * poly)
    return np.where(x >= 0, r, 2 - r)


def norm_cdf(z):
    return 0.5 * erfc(-np.asarray(z, dtype=float) / SQRT2)


def norm_sf(z):
    return 0.5 * erfc(np.asarray(z, dtype=float) / SQRT2)


def norm_inv(p):
    if p <= 0:
        return -math.inf
    if p >= 1:
        return math.inf
    if p < 0.02425:
        q = math.sqrt(-2 * math.log(p))
        x = (((((_C[0] * q + _C[1]) * q + _C[2]) * q + _C[3]) * q + _C[4]) * q + _C[5]) / ((((_D[0] * q + _D[1]) * q + _D[2]) * q + _D[3]) * q + 1)
    elif p <= 1 - 0.02425:
        q = p - 0.5
        r = q * q
        x = ((((((_A[0] * r + _A[1]) * r + _A[2]) * r + _A[3]) * r + _A[4]) * r + _A[5]) * q) / (((((_B[0] * r + _B[1]) * r + _B[2]) * r + _B[3]) * r + _B[4]) * r + 1)
    else:
        q = math.sqrt(-2 * math.log(1 - p))
        x = -(((((_C[0] * q + _C[1]) * q + _C[2]) * q + _C[3]) * q + _C[4]) * q + _C[5]) / ((((_D[0] * q + _D[1]) * q + _D[2]) * q + _D[3]) * q + 1)
    e = float(norm_cdf(x)) - p
    u = e * math.sqrt(2 * math.pi) * math.exp((x * x) / 2)
    return x - u / (1 + (x * u) / 2)


def norm_isf(p):
    """상단 꼬리확률 → z (배열, 아주 작은 p도 정밀하게)"""
    p = np.clip(np.atleast_1d(np.asarray(p, dtype=float)), 1e-300, 1 - 1e-16)
    x = np.empty_like(p)
    lo, hi = p < 0.02425, p > 1 - 0.02425
    mid = ~(lo | hi)
    q = np.sqrt(-2 * np.log(p[lo]))
    x[lo] = (((((_C[0] * q + _C[1]) * q + _C[2]) * q + _C[3]) * q + _C[4]) * q + _C[5]) / ((((_D[0] * q + _D[1]) * q + _D[2]) * q + _D[3]) * q + 1)
    q = p[mid] - 0.5
    r = q * q
    x[mid] = ((((((_A[0] * r + _A[1]) * r + _A[2]) * r + _A[3]) * r + _A[4]) * r + _A[5]) * q) / (((((_B[0] * r + _B[1]) * r + _B[2]) * r + _B[3]) * r + _B[4]) * r + 1)
    q = np.sqrt(-2 * np.log(1 - p[hi]))
    x[hi] = -(((((_C[0] * q + _C[1]) * q + _C[2]) * q + _C[3]) * q + _C[4]) * q + _C[5]) / ((((_D[0] * q + _D[1]) * q + _D[2]) * q + _D[3]) * q + 1)
    with np.errstate(all='ignore'):
        e = norm_cdf(x) - p
        u = e * math.sqrt(2 * math.pi) * np.exp((x * x) / 2)
        refined = x - u / (1 + (x * u) / 2)
    return -np.where(np.isfinite(refined) & (p > 1e-250), refined, x)


def lgamma(x):
    x = np.asarray(x, dtype=float)
    y = x.copy()
    tmp = x + 5.5 - (x + 0.5) * np.log(x + 5.5)
    ser = 1.000000000190015
    for g in _LG:
        y = y + 1
        ser = ser + g / y
    return -tmp + np.log((2.5066282746310005 * ser) / x)


def betacf(a, b, x):
    a, b, x = np.broadcast_arrays(*(np.asarray(v, dtype=float) for v in (a, b, x)))
    MAXIT, EPS, FPMIN = 300, 3e-14, 1e-300
    qab, qap, qam = a + b, a + 1, a - 1
    floor = lambda v: np.where(np.abs(v) < FPMIN, FPMIN, v)
    c = np.ones_like(x)
    d = 1 / floor(1 - (qab * x) / qap)
    h_ = d.copy()
    live = np.ones(x.shape, dtype=bool)
    with np.errstate(all='ignore'):
        for m in range(1, MAXIT + 1):
            m2 = 2 * m
            aa = (m * (b - m) * x) / ((qam + m2) * (a + m2))
            d1 = 1 / floor(1 + aa * d)
            c1 = floor(1 + aa / c)
            h1 = h_ * (d1 * c1)
            aa = (-(a + m) * (qab + m) * x) / ((a + m2) * (qap + m2))
            d1 = 1 / floor(1 + aa * d1)
            c1 = floor(1 + aa / c1)
            de = d1 * c1
            h1 = h1 * de
            c, d, h_ = np.where(live, c1, c), np.where(live, d1, d), np.where(live, h1, h_)
            live &= ~(np.abs(de - 1) < EPS)
            if not live.any():
                break
    return h_


def ibeta(x, a, b):
    x, a, b = np.broadcast_arrays(*(np.atleast_1d(np.asarray(v, dtype=float)) for v in (x, a, b)))
    out = np.where(x <= 0, 0.0, 1.0)
    mid = (x > 0) & (x < 1)
    if mid.any():
        xm, am, bm = x[mid], a[mid], b[mid]
        bt = np.exp(lgamma(am + bm) - lgamma(am) - lgamma(bm) + am * np.log(xm) + bm * np.log(1 - xm))
        front = xm < (am + 1) / (am + bm + 2)
        r = np.empty_like(xm)
        if front.any():
            r[front] = (bt[front] * betacf(am[front], bm[front], xm[front])) / am[front]
        if (~front).any():
            r[~front] = 1 - (bt[~front] * betacf(bm[~front], am[~front], 1 - xm[~front])) / bm[~front]
        out[mid] = r
    return out


def t_sf(t, df):
    t, df = np.atleast_1d(np.asarray(t, dtype=float)), np.atleast_1d(np.asarray(df, dtype=float))
    p = 0.5 * ibeta(df / (df + t * t), df / 2, 0.5)
    return np.where(t >= 0, p, 1 - p)


def f_sf(f, d1, d2):
    f = np.atleast_1d(np.asarray(f, dtype=float))
    with np.errstate(all='ignore'):
        p = ibeta(d2 / (d2 + d1 * f), d2 / 2, d1 / 2)
    return np.where(f <= 0, 1.0, p)


def var_of(n, s, ss):
    return max(0.0, (ss - (s * s) / n) / (n - 1)) if n > 1 else 0.0


def anova(n, s, ss):
    keep_ = n > 0
    n, s, ss = n[keep_].tolist(), s[keep_].tolist(), ss[keep_].tolist()
    k, N_ = len(n), sum(n)
    if k < 2 or N_ - k < 1:
        return None
    grand = sum(s) / N_
    ssb = sum(ni * (si / ni - grand) ** 2 for ni, si in zip(n, s))
    ssw = sum(ssi - ni * (si / ni) ** 2 for ni, si, ssi in zip(n, s, ss))
    if ssw <= 0:
        return None
    d1, d2 = k - 1, N_ - k
    F = ssb / d1 / (ssw / d2)
    return {'F': F, 'd1': d1, 'd2': d2, 'p': float(f_sf(F, d1, d2)[0])}


def welch(na, sa, ssa, nb, sb, ssb):
    """a가 b보다 큰지 단측 Welch t (요약통계 배열)"""
    na, sa, ssa, nb, sb, ssb = (np.atleast_1d(np.asarray(v, dtype=float)) for v in (na, sa, ssa, nb, sb, ssb))
    t, df, p = np.zeros_like(na), np.ones_like(na), np.ones_like(na)
    ok = (na >= 2) & (nb >= 2)
    with np.errstate(all='ignore'):
        va = np.maximum(0, (ssa - (sa * sa) / na) / (na - 1)) / na
        vb = np.maximum(0, (ssb - (sb * sb) / nb) / (nb - 1)) / nb
        se = np.sqrt(va + vb)
        ok &= se != 0
        t[ok] = (sa[ok] / na[ok] - sb[ok] / nb[ok]) / se[ok]
        df[ok] = (va[ok] + vb[ok]) ** 2 / ((va[ok] * va[ok]) / (na[ok] - 1) + (vb[ok] * vb[ok]) / (nb[ok] - 1))
    if ok.any():
        p[ok] = t_sf(t[ok], df[ok])
    return t, df, p


def quantile(v, p):
    if not len(v):
        return math.nan
    h_ = (len(v) - 1) * p
    lo = math.floor(h_)
    return float(v[lo] + (h_ - lo) * (v[min(lo + 1, len(v) - 1)] - v[lo]))

In [ ]:
def combo_key(s, items):
    return f'{s}|' + '|'.join(f'{q}:{u}' for q, u in items)


def binom_fix(n, x, p0):
    """bad(0/1) z의 꼬리 보정 배수. bad 비율이 낮고 웨이퍼가 적으면 bad 장수 분포가 오른쪽으로 길어서
    정규 근사 z가 실제보다 커진다. 정확한 이항 꼬리확률 P(bad ≥ x)로 구한 z ÷ 정규 근사 z를 곱해 줄인다(1보다 크게 하지는 않음)"""
    n, x = np.asarray(n, dtype=float), np.rint(np.asarray(x, dtype=float))
    z_norm = (x - n * p0) / np.sqrt(n * p0 * (1 - p0))
    fix = np.ones_like(n)
    up = z_norm > 0.5
    if up.any() and 0 < p0 < 1:
        nu, xu = n[up], x[up]
        ge = lambda k: np.where(k <= 0, 1.0, ibeta(np.full_like(nu, p0), np.maximum(k, 1), np.maximum(nu - k + 1, 1e-9)))
        fix[up] = np.clip(np.minimum(norm_isf(np.maximum(ge(xu), 1e-300)), 38.0) / z_norm[up], 0.0, 1.0)
    return fix


def shrink(n, N):
    """√(1 − 웨이퍼 수 ÷ 전체 웨이퍼 수). 전체 평균에는 조합의 웨이퍼도 들어 있어서, 조합 평균이 전체 평균에서
    우연히 벗어나는 폭은 σ/√n보다 이만큼 좁다. z를 이 값으로 나누면 조합을 나머지 웨이퍼와 비교한 z가 된다"""
    return np.sqrt(np.clip(1 - np.asarray(n, dtype=float) / N, 1 / N, None))


def analyze(data, min_n=20, max_depth=3, full_depth=2):
    """조합 생성과 조합별 지표 (웹 js/analysis.js analyze와 같은 조합 · 관문 로직).
    웨이퍼(root_lot_id + wafer_id) 한 장을 하나의 표본으로 보고, 신호(Value, bad)마다 조합을 나머지 웨이퍼와 비교한
    z = (조합 평균 − 전체 평균) ÷ (표준편차 ÷ √웨이퍼 수) ÷ √(1 − 웨이퍼 수 ÷ 전체 웨이퍼 수)를 구해 결합 z = (z_Value + z_bad) ÷ √(2 + 2r)로 판정한다.
    bad z는 정확한 이항 꼬리확률로 맞추고, r은 두 z가 우연히 같이 움직이는 정도(Order 1개 조합에서 잰 상관, 0~0.95)라
    같은 정보를 두 번 세지 않게 한다. good_bad가 없으면 z_Value만 쓴다"""
    N, steps, assign = data['N'], data['steps'], data['assign']
    sig = {'y': data['Y']} | ({'b': data['B']} if data.get('B') is not None else {})
    base = {k_: (float(v.mean()), float(v.std(ddof=1))) for k_, v in sig.items()}
    Y = sig['y']
    Y2 = Y * Y
    analysis_y = data.get('analysis_Y')  # 표시 평균만 계산. sig/base/부모 z에는 넣지 않는다.
    mu, sd = base['y']
    bad01 = np.nan_to_num(data['bad']) if 'b' in sig else None
    p0 = float(bad01.mean()) if 'b' in sig else 0.0

    def cell_keys(s, qs):
        A = assign[s][list(qs)]
        dims = [len(steps[s]['seqs'][q]['units']) for q in qs]
        ok = np.all(A != MISSING, axis=0)
        key = np.zeros(N, dtype=np.int64)
        for i, d in enumerate(dims):
            key = key * d + A[i]
        return dims, ok, key

    def cells(s, qs):
        """오더 묶음 qs에서 웨이퍼가 실제로 지난 칸(유닛 조합)만: 칸 번호(오름차순) · 칸마다 웨이퍼 수 · 웨이퍼 → 칸 순번.
        가능한 칸 수(유닛 수의 곱)가 아무리 커도 웨이퍼 수만큼만 계산한다"""
        dims, ok, key = cell_keys(s, qs)
        values = key[ok]
        space = math.prod(dims)
        # 작은 정수 공간은 정렬 대신 직접 집계. 원본과 같은 오름차순 u / inv / cnt를 반환한다.
        if space <= 1_000_000 and space <= max(4096, 4 * len(values)):
            dense_counts = np.bincount(values, minlength=space)
            u = np.flatnonzero(dense_counts)
            cnt = dense_counts[u]
            lookup = np.empty(space, dtype=np.intp)
            lookup[u] = np.arange(len(u), dtype=np.intp)
            inv = lookup[values]
        else:
            u, inv, cnt = np.unique(values, return_inverse=True, return_counts=True)
            inv = np.asarray(inv, dtype=np.intp).reshape(-1)
        return dims, ok, u, inv, cnt

    def stats(ok, inv, m, idx):
        """칸마다 신호별 합을 구해 idx 칸만 돌려준다"""
        w = lambda v: np.bincount(inv, weights=v[ok], minlength=m)[idx]
        out = {'ss': w(Y2)} | {f'sum_{k_}': w(v) for k_, v in sig.items()}
        if analysis_y is not None:
            out['sum_analysis'] = w(analysis_y)
        if 'b' in sig:
            out['x_bad'] = w(bad01)                   # bad 장수 (0/1): 이항분포 꼬리 보정용
        return out

    def sig_z(k_, nn, col):
        """조합들의 신호 k_ z: (조합 평균 − 전체 평균) ÷ (표준편차 ÷ √웨이퍼 수) ÷ √(1 − 웨이퍼 수 ÷ 전체 웨이퍼 수)"""
        m_, s_ = base[k_]
        return (col(f'sum_{k_}') / nn - m_) / (s_ / np.sqrt(nn)) / shrink(nn, N)

    def decode(qs, dims, key):
        units = []
        for d in reversed(dims):
            units.append(key % d)
            key //= d
        return tuple((q, int(u)) for q, u in zip(qs, reversed(units)))

    def parent_keys(dims, keys, i):
        """칸 번호에서 i번째 오더 자리를 뺀 부모 칸 번호"""
        low = math.prod(dims[i + 1:])
        return (keys // (low * dims[i])) * low + keys % low

    seq_info = []
    for s, st in enumerate(steps):
        rows = []
        for q, sq in enumerate(st['seqs']):
            a = assign[s][q]
            ok, nu = a != MISSING, len(sq['units'])
            n_u = np.bincount(a[ok], minlength=nu)
            s_u, ss_u = np.bincount(a[ok], weights=Y[ok], minlength=nu), np.bincount(a[ok], weights=Y2[ok], minlength=nu)
            units = [{'name': name, 'n': int(n_u[u]), 'mean': s_u[u] / n_u[u] if n_u[u] else math.nan,
                      'sd': math.sqrt(var_of(int(n_u[u]), s_u[u], ss_u[u]))} for u, name in enumerate(sq['units'])]
            testable = nu >= 2
            rows.append({'name': sq['name'], 'units': units, 'testable': testable, 'anova': anova(n_u, s_u, ss_u) if testable else None})
        seq_info.append(rows)

    combos, by_key, r = [], {}, 0.0
    flagged = {}                        # (스텝, 오더 묶음) → z 1.96을 넘은 칸 번호들 (다음 깊이의 관문)
    has_big = set()                     # 웨이퍼 MIN_N장 이상인 칸이 있는 (스텝, 오더 묶음). 부모에 없으면 자식에도 없다
    multi = [[q for q, sq in enumerate(st['seqs']) if len(sq['units']) >= 2] for st in steps]
    search = [0] * (max_depth + 1)
    progress_time = time.perf_counter()
    for k in range(1, max_depth + 1):
        gated = k > full_depth
        cand = []
        for s in range(len(steps)):
            if len(multi[s]) < k:
                continue
            for qs in combinations(multi[s], k):
                if SHOW_PROGRESS and time.perf_counter() - progress_time >= 5:
                    print(f'    깊이 {k}/{max_depth} · STEP {s+1}/{len(steps)} · 후보 {len(combos):,}개', flush=True)
                    progress_time = time.perf_counter()
                subs = [(s, qs[:i] + qs[i + 1:]) for i in range(k)]
                if k > 1 and not all(p in has_big for p in subs):
                    continue                    # 오더 하나를 뺀 묶음에 큰 칸이 없으면 이 묶음에도 없다
                dims, ok, u, inv, cnt = cells(s, qs)
                big = np.flatnonzero(cnt >= min_n)
                search[k] += len(big)           # 기준 z의 보정에는 계산을 건너뛴 칸도 센다
                if len(big):
                    has_big.add((s, qs))
                if not len(big) or (gated and not any(p in flagged for p in subs)):
                    continue
                if gated:                       # 오더 하나를 뺀 부모 조합 중 하나라도 z 1.96을 넘은 칸만 계산
                    keep = np.zeros(len(big), dtype=bool)
                    for i, p in enumerate(subs):
                        if p in flagged:
                            keep |= np.isin(parent_keys(dims, u[big], i), flagged[p])
                    big = big[keep]
                    if not len(big):
                        continue
                cand.append((s, qs, dims, u[big], cnt[big], stats(ok, inv, len(u), big)))
        if not cand:
            continue
        col = lambda f: np.concatenate([c[5][f] for c in cand]).astype(float)
        nn = np.concatenate([c[4] for c in cand]).astype(float)
        sm, ss = col('sum_y'), col('ss')
        analysis_means = col('sum_analysis') / nn if analysis_y is not None else None
        zy = sig_z('y', nn, col)
        zb = sig_z('b', nn, col) * binom_fix(nn, col('x_bad'), p0) if 'b' in sig else None
        if zb is not None and k == 1 and len(zy) > 2:   # 두 z의 상관: 오더 1개 조합(대부분 우연)에서 잰다
            r = float(np.clip(np.corrcoef(zy, zb)[0, 1], 0.0, 0.95))
        z = zy if zb is None else (zy + zb) / math.sqrt(2 + 2 * r)
        bad_rate = col('x_bad') / nn if 'b' in sig else None   # 원래(0/1) bad 비율 — 랭킹의 초과 bad 웨이퍼 계산용
        j = 0
        for s, qs, dims, keys, _, _ in cand:
            hot = []
            for key in keys.tolist():
                items = decode(qs, dims, key)
                c = {'id': len(combos), 'key': combo_key(s, items), 'step': s, 'k': k, 'items': items,
                     'n': int(nn[j]), 'sum': float(sm[j]), 'ss': float(ss[j]), 'mean': float(sm[j] / nn[j]),
                     'z': float(z[j]), 'z_y': float(zy[j]), 'z_b': float(zb[j]) if zb is not None else math.nan,
                     'bad_rate': float(bad_rate[j]) if bad_rate is not None else math.nan}
                if analysis_means is not None:
                    c['analysis_mean'] = float(analysis_means[j])
                combos.append(c)
                by_key[c['key']] = c
                if c['z'] > Z95:
                    hot.append(key)
                j += 1
            if hot:
                flagged[(s, qs)] = np.array(hot, dtype=np.int64)
    depth_count = [0] * (max_depth + 1)
    for c in combos:
        depth_count[c['k']] += 1
    return {'mu': mu, 'sd': sd, 'N': N, 'max_depth': max_depth, 'signals': list(sig), 'r': r, 'p0': p0 if 'b' in sig else None,
            'seq_info': seq_info, 'combos': combos, 'by_key': by_key, 'depth_count': depth_count, 'search_space': search}



In [ ]:
POINT_COLUMNS = ['part_id', 'step', 'depth', 'path', 'n', 'sample_mean']

def mean_envelope(points, mu_hat, sigma_hat, *, sigma_hat_raw):
    # 선택한 분석 단위의 웨이퍼 표준편차를 모든 n에 공통으로 사용한다.
    # sigma_hat_raw는 원단위 참고값이며 ON의 경계 계산에는 사용하지 않는다.
    summary_columns = ['n', 'm_n', 'mean_of_means', 'sigma_hat', 'sigma_hat_raw', 'sigma_hat_n',
                       'mu_hat', 'sqrt_2log_m', 'half_width', 'lower', 'upper']
    if points.empty:
        summary = pd.DataFrame(columns=summary_columns)
        result = points.copy()
        for col in summary_columns:
            if col not in ('n', 'mean_of_means'):
                result[col] = pd.Series(dtype=float)
        result['outside'] = pd.Series(dtype='boolean')
        result['direction'] = pd.Series(dtype=str)
        return result, summary
    summary = (points.groupby('n', sort=True)['sample_mean']
               .agg(m_n='size', mean_of_means='mean')
               .reset_index())
    summary['sigma_hat_raw'] = sigma_hat_raw
    summary['sigma_hat'] = sigma_hat
    summary['sigma_hat_n'] = sigma_hat / np.sqrt(summary['n'])
    summary['mu_hat'] = mu_hat
    summary['sqrt_2log_m'] = np.sqrt(2 * np.log(summary['m_n']))
    # m_n=1도 웨이퍼 표준편차로 표준오차를 계산할 수 있다. 이때 log(1)=0으로 폭은 0이다.
    half_width = summary['sigma_hat_n'] * summary['sqrt_2log_m']
    summary['half_width'] = half_width
    summary['lower'] = mu_hat - half_width
    summary['upper'] = mu_hat + half_width
    result = points.merge(summary.drop(columns='mean_of_means'), on='n', how='left', validate='many_to_one')
    # 경계와 소수점 연산 오차 수준으로 같은 값은 경계 위로 취급한다. 경계 수식 자체는 바꾸지 않는다.
    scale = result[['sample_mean', 'lower', 'upper']].abs().max(axis=1)
    rounding_tolerance = 32 * np.finfo(float).eps * scale
    above = result['sample_mean'] > result['upper'] + rounding_tolerance
    below = result['sample_mean'] < result['lower'] - rounding_tolerance
    result['outside'] = (above | below).astype('boolean')
    result['direction'] = np.select([above, below], ['위쪽', '아래쪽'], default='경계 안')
    return result, summary

def step_sort_key(value):
    return (0, float(value), value) if value.replace('.', '', 1).isdigit() else (1, 0.0, value)


def prepare_reference_part(part_raw, part_id):
    # 원본처럼 first()는 결측을 건너뛴 첫 값을 사용한다.
    wafer_all = (part_raw.group_by(WAFER_KEY).agg(
        pl.col('_y').drop_nulls().first().alias('raw_y'),
        pl.col('_y').drop_nulls().n_unique().alias('y_n'),
        pl.col('_bad').drop_nulls().first().alias('bad'),
        pl.col('_gb').drop_nulls().first().alias('gb'),
    ).sort(WAFER_KEY))
    if wafer_all['raw_y'].null_count():
        raise ValueError(f'part {part_id}: raw y가 없는 웨이퍼 {wafer_all["raw_y"].null_count()}장. 입력을 확인하세요.')
    has_bad = wafer_all['bad'].null_count() < wafer_all.height
    wafer_used = wafer_all.filter(pl.col('bad').is_not_null()) if has_bad else wafer_all
    wafer_used = wafer_used.with_row_index('_wi')
    n = wafer_used.height
    raw_y = wafer_used['raw_y'].to_numpy()
    if n < 2 * MIN_N:
        raise ValueError(f'원본 최소 비교 조건 미달: 웨이퍼 {n}장 < {2 * MIN_N}장')
    if not float(np.std(raw_y)) > 0:
        raise ValueError('원본 비교 조건 미달: raw y 표준편차가 0입니다.')
    history = part_raw.filter(pl.col('_step').is_not_null() & pl.col('_order').is_finite() & pl.col('eqp_id').is_not_null())
    history = history.join(wafer_used.select(WAFER_KEY + ['_wi']), on=WAFER_KEY, how='inner')
    before = history.height
    history = (history.sort(['_time', '_row'], nulls_last=(REWORK != 'last'))
               .unique(subset=['_wi', '_step', '_order'], keep=REWORK, maintain_order=True))
    duplicates = before - history.height
    if history.is_empty():
        raise ValueError('유효한 공정 이력이 없습니다.')
    # 유닛 이름·정렬 순서는 원본 eqp_id-chamber_id 방식과 같다.
    history = history.with_columns(pl.when(pl.col('chamber_id') != '')
        .then(pl.concat_str(['eqp_id', 'chamber_id'], separator='-'))
        .otherwise(pl.col('eqp_id')).alias('_unit'))
    step_groups = history.partition_by('_step', as_dict=True)
    steps, assign = [], []
    for step_key in sorted(step_groups, key=lambda key: step_sort_key(str(key[0]))):
        name, step_history = str(step_key[0]), step_groups[step_key]
        orders = sorted(step_history['_order'].unique().to_list())
        a = np.full((len(orders), n), MISSING, dtype=np.uint16)
        seqs = []
        for qi, order in enumerate(orders):
            h = step_history.filter(pl.col('_order') == order)
            units = sorted(h['_unit'].unique().to_list())
            if len(units) >= MISSING:
                raise ValueError('한 Order의 유닛 수가 너무 많습니다.')
            codes = h['_unit'].replace_strict(dict(zip(units, range(len(units)))), return_dtype=pl.UInt16).to_numpy()
            a[qi, h['_wi'].to_numpy()] = codes
            label = str(int(order)) if float(order).is_integer() else str(order)
            seqs.append({'name': f'O{label}', 'ord': float(order), 'units': units})
        steps.append({'index': len(steps), 'name': name, 'seqs': seqs})
        assign.append(a)
    bad = wafer_used['bad'].to_numpy()
    b = bad.copy() if has_bad else None
    if b is not None and b.std() == 0:
        b = None
    data = {'N': n, 'Y': raw_y.copy() if HIGHER_IS_WORSE else -raw_y,
            'B': b, 'bad': bad, 'steps': steps, 'assign': assign}
    wf = pd.DataFrame(wafer_used.select(WAFER_KEY + ['raw_y']).to_dict(as_series=False)).set_index(WAFER_KEY)
    quality = {'wafer_total': wafer_all.height, 'excluded_non_gb': wafer_all.height - n,
               'multiple_y': int((wafer_all['y_n'] > 1).sum()), 'duplicate_history_rows': duplicates}
    return wf, data, quality


"""Robust lambda fitting for Yeo–Johnson applied directly to raw observations.

Independent NumPy/SciPy implementation based on Raymaekers & Rousseeuw,
Transforming variables to central normality, doi:10.1007/s10994-021-05960-5,
Sections 2.1–2.4. Unlike their recommended prestandardized setup, the input
here is NOT centered/scaled before Yeo–Johnson. Rectification is used only
in the initial fit. The final transformation is ordinary Yeo–Johnson.

The authors' R cellWise implementation is GPL >= 2; no R/C++ source is
vendored or translated here. Numerical choices are documented below.
"""
import numpy as np


def _yj_huber_location_scale(values):
    """One-step Huber M location and scale, initialized by median / normal MAD.

    k=1.5 for both estimates. The scale consistency constant is evaluated
    analytically as E[min(Z**2, k**2)] for Z standard normal. These internal
    robust estimates fit lambda only; they are not the chart's mean / SD.
    """
    from scipy.special import ndtr, ndtri

    z = np.asarray(values, dtype=float)
    if z.ndim != 1 or not np.isfinite(z).all():
        raise ValueError("Yeo-Johnson fit encountered non-finite transformed values")
    location = float(np.median(z))
    scale0 = float(np.median(np.abs(z - location))) / ndtri(0.75)
    if not np.isfinite(scale0) or scale0 <= 0:
        raise ValueError("MAD is zero: robust lambda fitting is undefined for this part")
    k = 1.5
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        residual = (z - location) / scale0
        weights = np.minimum(1.0, k / np.maximum(np.abs(residual), np.finfo(float).tiny))
        location += scale0 * float(np.mean(np.clip(residual, -k, k))) / float(weights.mean())
    if not np.isfinite(location):
        raise ValueError("Huber location is numerically non-finite")
    residual_abs = np.abs(z - location)
    scale0 = float(np.median(residual_abs)) / ndtri(0.75)
    if not np.isfinite(scale0) or scale0 <= 0:
        raise ValueError("MAD is zero after Huber location fitting")
    # Integral z^2 phi(z) on [-k,k], plus the two winsorized tails.
    tail = float(ndtr(-k))
    density = float(np.exp(-0.5 * k * k) / np.sqrt(2 * np.pi))
    normal_factor = 1 - 2 * (k * density + tail) + 2 * k * k * tail
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        clipped_residual = np.minimum(residual_abs / scale0, k)
        scale = scale0 * np.sqrt(float(np.mean(clipped_residual**2)) / normal_factor)
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError("Huber scale is zero or numerically non-finite")
    return location, float(scale)


def _yj_rectified_for_fit(x_sorted, lmbda, quartiles):
    """Paper equations (9),(10), with quartile knots restricted to their branch.

    Cl=min(Q1,0), Cu=max(Q3,0) allows uncentered raw input while retaining
    the required signs of the rectification knots. No data is removed.
    """
    from scipy.stats import yeojohnson

    low, high = quartiles
    if lmbda == 1:
        return x_sorted.copy()
    knot = high if lmbda < 1 else low
    linear = x_sorted > knot if lmbda < 1 else x_sorted < knot
    result = np.empty_like(x_sorted)
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        result[~linear] = yeojohnson(x_sorted[~linear], lmbda=lmbda)
        knot_y = float(yeojohnson(np.array([knot]), lmbda=lmbda)[0])
        derivative = np.exp((lmbda - 1) * np.sign(knot) * np.log1p(abs(knot)))
        result[linear] = knot_y + derivative * (x_sorted[linear] - knot)
    return result


def _yj_minimize_bounded(objective, bounds, *, stage):
    """Coarse grid plus local refinements; fail visibly if optimum hits bounds.

    Bounds are numerical search limits, not an assertion about allowed YJ
    lambda. Callers can widen them or handle the exception explicitly.
    """
    from scipy.optimize import minimize_scalar

    lower, upper = map(float, bounds)
    if not np.isfinite([lower, upper]).all() or lower >= upper:
        raise ValueError("Yeo-Johnson lambda bounds must be finite and increasing")
    grid = np.unique(np.r_[np.linspace(lower, upper, 21),
                            [v for v in (0.0, 1.0, 2.0) if lower <= v <= upper]])
    scores = np.array([objective(value) for value in grid])
    finite = np.isfinite(scores)
    if not finite.any():
        raise ValueError(f"{stage}: no numerically valid lambda in {bounds}")
    candidates = [(float(scores[i]), float(grid[i])) for i in np.flatnonzero(finite)]
    # Refine the three best grid neighborhoods to reduce dependence on a
    # single initial bracket in the bounded, potentially nonconvex QQ loss.
    for index in np.argsort(scores)[:3]:
        if not np.isfinite(scores[index]):
            continue
        left, right = grid[max(0, index - 1)], grid[min(len(grid) - 1, index + 1)]
        if left == right:
            continue
        # Invalid lambda candidates deliberately return infinity. Brent's
        # interpolation may perform intermediate inf arithmetic; candidate
        # validation below still rejects non-finite results explicitly.
        with np.errstate(invalid="ignore", over="ignore"):
            fit = minimize_scalar(objective, bounds=(left, right), method="bounded",
                                  options={"xatol": 1e-5, "maxiter": 100})
        if fit.success and np.isfinite(fit.fun):
            candidates.append((float(fit.fun), float(fit.x)))
    score, estimate = min(candidates)
    margin = max(1e-4, (upper - lower) * 1e-4)
    if estimate <= lower + margin or estimate >= upper - margin:
        raise ValueError(f"{stage}: lambda optimum reached search bounds {bounds}; widen bounds")
    return estimate, score


def _yj_minimize_expanding(objective, bounds, *, stage, max_expansions=12):
    """Expand an initial numerical bracket when the best fit reaches an edge.

    Raw rates can need |lambda| far greater than 4 or 6. Expansion changes
    the numerical search interval only, never the input units or location.
    """
    lower, upper = map(float, bounds)
    for attempt in range(max_expansions + 1):
        try:
            estimate, score = _yj_minimize_bounded(objective, (lower, upper), stage=stage)
            return estimate, score, (lower, upper)
        except ValueError as exc:
            if "reached search bounds" not in str(exc) or attempt == max_expansions:
                raise
            left_score, right_score = objective(lower), objective(upper)
            width = upper - lower
            if left_score <= right_score:
                lower -= 2 * width
            else:
                upper += 2 * width
    raise ValueError(f"{stage}: search interval expansion failed")


def fit_rewml_yj(raw_y, *, bounds=(-4.0, 6.0)):
    """Return (lambda, diagnostics) using a RewML-based raw-input adaptation.

    Uses all observations (no sampling), one initial sort, a bounded Tukey
    QQ loss (c=.5), followed by exactly two hard-reweighted ordinary-YJ
    likelihood fits. The .995 normal cutoff selects observations used to
    estimate lambda; it does NOT define the extreme-value chart envelope.

    Only lambda fitting uses robust location/scale and zero/one weights.
    Apply the fitted ordinary YJ to EVERY original wafer afterward, and
    compute chart moments from ALL transformed wafers. Too-small samples,
    zero MAD (often due to many tied zeros), overflow, and unresolved boundary
    optima after adaptive expansion raise ValueError; the caller must report any fallback explicitly.
    """
    from scipy.special import ndtri
    from scipy.stats import yeojohnson

    raw = np.asarray(raw_y, dtype=float)
    if raw.ndim != 1 or len(raw) < 8 or not np.isfinite(raw).all():
        raise ValueError("RewML needs a finite one-dimensional array with at least 8 values")
    x = np.sort(raw)
    _yj_huber_location_scale(x)  # Explicitly reject zero-MAD / degenerate parts.
    quartiles = np.quantile(x, [0.25, 0.75])
    knots = (min(float(quartiles[0]), 0.0), max(float(quartiles[1]), 0.0))
    theoretical = ndtri((np.arange(1, len(x) + 1) - 1/3) / (len(x) + 1/3))
    objective_evaluations = 0

    def qq_objective(lmbda):
        nonlocal objective_evaluations
        objective_evaluations += 1
        try:
            transformed = _yj_rectified_for_fit(x, lmbda, knots)
            location, scale = _yj_huber_location_scale(transformed)
            with np.errstate(over="ignore", invalid="ignore"):
                residual = (transformed - location) / scale - theoretical
                square = np.minimum(np.abs(residual) / 0.5, 1.0)**2
                loss = float(np.mean(1 - (1 - square)**3))
            return loss if np.isfinite(loss) else np.inf
        except (ValueError, FloatingPointError, OverflowError):
            return np.inf

    lmbda, initial_loss, used_bounds = _yj_minimize_expanding(qq_objective, bounds, stage="RewML initial fit")
    initial_lmbda = lmbda
    # First hard weights use the rectified initialization. Further weights
    # use ordinary YJ after each likelihood update. This raw-input
    # adaptation uses rectified initial weights, as in cellWise transfo;
    # the weighted likelihood itself always uses ordinary YJ.
    transformed = _yj_rectified_for_fit(x, lmbda, knots)
    cutoff = float(ndtri(0.995))
    retained = []
    lambda_steps = []
    for step in range(2):
        location, scale = _yj_huber_location_scale(transformed)
        with np.errstate(over="ignore", invalid="ignore"):
            keep = np.abs((transformed - location) / scale) <= cutoff
        fit_x = x[keep]
        if len(fit_x) < 3 or fit_x[0] == fit_x[-1]:
            raise ValueError("RewML weights retained too few distinct observations")
        retained.append(int(keep.sum()))
        log_jacobian_mean = float(np.mean(np.sign(fit_x) * np.log1p(np.abs(fit_x))))

        def weighted_nll(candidate):
            nonlocal objective_evaluations
            objective_evaluations += 1
            with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
                values = yeojohnson(fit_x, lmbda=candidate)
                # Scale before variance to avoid overflow in squared values.
                factor = float(np.max(np.abs(values)))
                if not np.isfinite(factor) or factor == 0:
                    return np.inf
                variance_scaled = float(np.var(values / factor))
                if not np.isfinite(variance_scaled) or variance_scaled <= 0:
                    return np.inf
                log_variance = np.log(variance_scaled) + 2 * np.log(factor)
                value = 0.5 * log_variance - (candidate - 1) * log_jacobian_mean
            return float(value) if np.isfinite(value) else np.inf

        lmbda, _, used_bounds = _yj_minimize_expanding(weighted_nll, used_bounds, stage=f"RewML weighted step {step + 1}")
        lambda_steps.append(lmbda)
        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            transformed = yeojohnson(x, lmbda=lmbda)
    location, scale = _yj_huber_location_scale(transformed)
    with np.errstate(over="ignore", invalid="ignore"):
        final_keep = np.abs((transformed - location) / scale) <= cutoff
    return float(lmbda), {
        "method": "rewml_raw",
        "fit_n": int(len(x)),
        "initial_lambda": float(initial_lmbda),
        "lambda_initial": float(initial_lmbda),
        "initial_qq_loss": float(initial_loss),
        "lambda_steps": tuple(float(value) for value in lambda_steps),
        "kept_per_step": tuple(retained),
        "n_kept_last_fit": retained[-1],
        "n_used_last": retained[-1],
        "n_kept_final": int(final_keep.sum()),
        "initial_search_bounds": tuple(map(float, bounds)),
        "bounds_used": tuple(map(float, used_bounds)),
        "objective_evaluations": objective_evaluations,
        "prestandardized": False,
    }


def transform_part_y(raw_y, enabled, *, method=None, return_info=False):
    """λ 추정만 선택한다. 출력은 raw 입력에 대한 일반 YJ이며 표준화하지 않는다."""
    if not isinstance(enabled, (bool, np.bool_)):
        raise ValueError('use_yeo_johnson은 True 또는 False여야 합니다.')
    raw_y = np.asarray(raw_y, dtype=float)
    if raw_y.ndim != 1 or raw_y.size < 2 or not np.isfinite(raw_y).all():
        raise ValueError('변환 입력은 유한한 값이 2개 이상인 1차원 웨이퍼 배열이어야 합니다.')
    if np.all(raw_y == raw_y[0]):
        raise ValueError('raw y가 상수이므로 표준편차와 변환을 추정할 수 없습니다.')
    requested = YJ_LAMBDA_METHOD if method is None else method
    if requested not in ('robust', 'mle'):
        raise ValueError("lambda_method는 'robust' 또는 'mle'여야 합니다.")
    info = {'requested': requested if enabled else 'raw', 'used': 'raw',
            'status': 'off', 'note': '', 'n_fit': len(raw_y), 'n_used_last': len(raw_y)}
    if not enabled:
        result = (raw_y.copy(), None)
        return (*result, info) if return_info else result
    try:
        from scipy.stats import yeojohnson
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError('Yeo–Johnson ON에는 SciPy가 필요합니다. %pip install scipy 후 다시 실행하세요.') from exc
    info.update(used=requested, status='ok')
    try:
        if requested == 'robust':
            try:
                fitted_lambda, details = fit_rewml_yj(raw_y)
                info.update(details)
                values = yeojohnson(raw_y, lmbda=fitted_lambda)
            except (ValueError, RuntimeError, OverflowError, FloatingPointError) as exc:
                if YJ_ROBUST_FALLBACK == 'error':
                    raise ValueError(f'강건한 λ 추정 불가: {exc}') from exc
                info.update(used='mle', status='fallback', note=str(exc), n_used_last=len(raw_y))
                values, fitted_lambda = yeojohnson(raw_y)
        else:
            values, fitted_lambda = yeojohnson(raw_y)
    except (ValueError, RuntimeError, OverflowError, FloatingPointError) as exc:
        raise ValueError(f'Yeo–Johnson 적합 실패: {exc}') from exc
    if not np.isfinite(fitted_lambda) or not np.isfinite(values).all():
        raise ValueError('Yeo–Johnson 결과에 비유한 값이 있습니다. 입력 범위를 확인하세요.')
    sigma = float(np.std(values, ddof=1))
    if not np.isfinite(sigma) or sigma <= 0:
        raise ValueError('Yeo–Johnson 결과의 표준편차가 0 또는 비유한 값입니다.')
    result = (values, float(fitted_lambda))
    return (*result, info) if return_info else result


def rank_normal_part_y(raw_y):
    """Part별 고유 웨이퍼를 평균순위 정규점수로 변환. 동률/입력 순서 불변."""
    raw_y = np.asarray(raw_y, dtype=float)
    if raw_y.ndim != 1 or raw_y.size < 2 or not np.isfinite(raw_y).all():
        raise ValueError('순위 변환 입력은 유한한 값이 2개 이상인 1차원 배열이어야 합니다.')
    unique, counts = np.unique(raw_y, return_counts=True)
    if len(unique) < 2:
        raise ValueError('raw y가 상수이므로 순위 점수의 표준편차가 0입니다.')
    try:
        from scipy.stats import rankdata
        from scipy.special import ndtri
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError('순위 정규변환에는 SciPy가 필요합니다. %pip install scipy 후 다시 실행하세요.') from exc
    ranks = rankdata(raw_y, method='average')
    values = ndtri((ranks - 0.5) / len(raw_y))
    tied_fraction = float(counts[counts > 1].sum() / len(raw_y))
    max_tie_fraction = float(counts.max() / len(raw_y))
    note = (f'동일 y 최대 집단 {max_tie_fraction:.1%}: 동률에 같은 점수를 부여하므로 정규분포와 다를 수 있습니다.'
            if len(unique) < len(raw_y) else '')
    return values, {'requested': 'rank_normal', 'used': 'rank_normal', 'status': 'ok',
                    'note': note, 'n_fit': len(raw_y), 'n_used_last': len(raw_y),
                    'n_unique': len(unique), 'tied_fraction': tied_fraction,
                    'max_tie_fraction': max_tie_fraction, 'ties': 'average'}


def analysis_value_label(report):
    return {'raw': 'raw y', 'yeo-johnson': 'Yeo–Johnson y',
            'rank_normal': 'Rank-normal score'}[report['transform']]


def analysis_mode_label(report):
    if report['transform'] == 'raw':
        return 'Raw · transform OFF'
    if report['transform'] == 'rank_normal':
        return 'Rank-normal · average ties'
    fit = report['transform_fit']
    label = f"Yeo–Johnson {fit['used']} · λ={report['transform_lambda']:.6g}"
    return label + (' (fallback)' if fit['status'] == 'fallback' else '')


def analyze_part(part_raw, part_id, *, transform=None, use_yeo_johnson=None, lambda_method=None):
    start = time.perf_counter()
    # 이전의 명시적 use_yeo_johnson 호출도 지원한다.
    if use_yeo_johnson is not None:
        if transform is not None or not isinstance(use_yeo_johnson, (bool, np.bool_)):
            raise ValueError('transform과 use_yeo_johnson을 동시에 지정할 수 없으며, 후자는 bool이어야 합니다.')
        selected_transform = 'yeo_johnson' if use_yeo_johnson else 'raw'
    else:
        selected_transform = Y_TRANSFORM if transform is None else transform
    if selected_transform not in ('raw', 'yeo_johnson', 'rank_normal'):
        raise ValueError("transform은 'raw', 'yeo_johnson', 'rank_normal' 중 하나여야 합니다.")
    enabled = selected_transform != 'raw' 
    wf, data, quality = prepare_reference_part(part_raw, part_id)
    if selected_transform == 'rank_normal':
        analysis_y, lambda_info = rank_normal_part_y(wf['raw_y'].to_numpy())
        fitted_lambda = None
    else:
        analysis_y, fitted_lambda, lambda_info = transform_part_y(
            wf['raw_y'].to_numpy(), enabled, method=lambda_method, return_info=True)
    if lambda_info['status'] == 'fallback':
        print(f"  part {part_id}: 강건 λ → MLE 전환: {lambda_info['note']}", flush=True)
    wf['analysis_y'] = analysis_y
    transform = 'yeo-johnson' if selected_transform == 'yeo_johnson' else selected_transform
    if enabled:
        # data['Y']는 그대로 raw: 변환을 켜도 깊이 3+의 후보 선별이 바뀌지 않는다.
        data['analysis_Y'] = analysis_y
    if quality['multiple_y']:
        print(f"  raw y가 여러 개인 웨이퍼 {quality['multiple_y']:,}장: 원본과 같이 첫 값 사용")
    if SHOW_PROGRESS:
        mode = ('Rank-normal (평균 순위)' if transform == 'rank_normal' else
                f"Yeo–Johnson {lambda_info['used']} λ={fitted_lambda:.6g}" if enabled else 'raw (변환 OFF)')
        print(f"  {mode} · 웨이퍼 {data['N']:,}장 / STEP {len(data['steps']):,}개: raw 후보 생성 시작", flush=True)
    if transform == 'rank_normal' and lambda_info['note'] and SHOW_PROGRESS:
        print(f"  {lambda_info['note']}", flush=True)
    reference = analyze(data, min_n=MIN_N, max_depth=MAX_DEPTH, full_depth=2)
    sign = 1 if HIGHER_IS_WORSE else -1
    target_step = (re.sub(STEP_SUFFIX, '', str(STEP)) if STEP_SUFFIX else str(STEP)) if STEP is not None else None
    rows = []
    for c in reference['combos']:
        step = data['steps'][c['step']]
        if target_step is not None and step['name'] != target_step:
            continue
        path = ' → '.join(f"{step['seqs'][q]['name']}: {step['seqs'][q]['units'][u]}" for q, u in c['items'])
        raw_mean = sign * c['mean']
        rows.append({'part_id': part_id, 'step': step['name'], 'depth': c['k'],
                     'path': path, 'n': c['n'], 'raw_mean': raw_mean,
                     'sample_mean': c['analysis_mean'] if enabled else raw_mean, 'combo_key': c['key']})
    points = pd.DataFrame(rows, columns=POINT_COLUMNS + ['raw_mean', 'combo_key'])
    mu_hat_raw = float(wf['raw_y'].mean())
    sigma_hat_raw = float(wf['raw_y'].std(ddof=1))
    mu_hat = float(wf['analysis_y'].mean())
    sigma_hat = float(wf['analysis_y'].std(ddof=1))
    points, envelope = mean_envelope(points, mu_hat, sigma_hat, sigma_hat_raw=sigma_hat_raw)
    # 같은 조합에서 raw 경계와 분석 경계의 판정을 비교한다. 후보 생성은 재실행하지 않는다.
    if enabled:
        raw_points = points[['n', 'raw_mean']].rename(columns={'raw_mean': 'sample_mean'})
        raw_result, raw_envelope = mean_envelope(raw_points, mu_hat_raw, sigma_hat_raw,
                                                sigma_hat_raw=sigma_hat_raw)
    else:
        raw_result, raw_envelope = points, envelope.copy()
    for name in ('lower', 'upper', 'outside', 'direction'):
        points[f'raw_{name}'] = raw_result[name].to_numpy()
    active_out = points['outside'].fillna(False).to_numpy(bool)
    raw_out = points['raw_outside'].fillna(False).to_numpy(bool)
    points['detection_comparison'] = np.select(
        [raw_out & active_out, raw_out, active_out],
        ['양쪽 경계 밖', 'raw만 경계 밖', '분석값만 경계 밖'], default='양쪽 경계 안')
    points['raw_mean_difference'] = points['raw_mean'] - mu_hat_raw
    points['analysis_mean_difference'] = points['sample_mean'] - mu_hat
    raw_envelope['transform'] = 'raw'
    raw_envelope['transform_lambda'] = None
    raw_envelope['mu_hat_raw'] = mu_hat_raw
    for table in (points, envelope):
        table['transform'] = transform
        table['transform_lambda'] = fitted_lambda
        table['mu_hat_raw'] = mu_hat_raw
        table['transform_fit_method'] = lambda_info['used']
        table['transform_fit_status'] = lambda_info['status']
        table['transform_fit_note'] = lambda_info['note']
        is_yj = transform == 'yeo-johnson'
        table['lambda_method_requested'] = lambda_info['requested'] if is_yj else None
        table['lambda_method_used'] = lambda_info['used'] if is_yj else None
        table['lambda_fit_status'] = lambda_info['status'] if is_yj else 'not_applicable'
        table['lambda_fit_note'] = lambda_info['note'] if is_yj else ''
        table['lambda_fit_n_used'] = lambda_info['n_used_last'] if is_yj else None
        table['rank_max_tie_fraction'] = lambda_info.get('max_tie_fraction', np.nan)
        table['rank_n_unique'] = lambda_info.get('n_unique', np.nan)
    return {'part_id': part_id, 'wafer_values': wf, 'mu_hat': mu_hat, 'mu_hat_raw': mu_hat_raw,
            'sigma_hat': sigma_hat, 'sigma_hat_raw': sigma_hat_raw, 'points': points,
            'transform': transform, 'transform_lambda': fitted_lambda,
            'transform_fit': lambda_info, 'lambda_fit': lambda_info,
            'envelope': envelope, 'raw_envelope': raw_envelope, 'quality': quality, 'depth_count': reference['depth_count'],
            'search_space': reference['search_space'], 'seconds': time.perf_counter() - start}


In [ ]:
results_by_part = {}
rows = []
# 제품마다 전체 CSV를 다시 스캔하지 않도록 한 번에 분할한다.
part_groups = raw.partition_by('part_id', as_dict=True)
for part_id in selected_parts:
    if SHOW_PROGRESS:
        print(f'part {part_id} 분석 시작', flush=True)
    try:
        report = analyze_part(part_groups.pop((part_id,)), part_id)
    except ValueError as exc:
        print(f'  분석 제외: {exc}', flush=True)
        rows.append({'part_id': part_id, '상태': str(exc)})
        continue
    results_by_part[part_id] = report
    points, quality = report['points'], report['quality']
    raw_out = points['raw_outside'].fillna(False).to_numpy(bool)
    active_out = points['outside'].fillna(False).to_numpy(bool)
    rows.append({'part_id': part_id, '상태': '완료', '분석 웨이퍼': len(report['wafer_values']),
                 'G/B 외 제외 웨이퍼': quality['excluded_non_gb'], 'raw y 평균': report['mu_hat_raw'],
                 'raw y 표준편차': report['sigma_hat_raw'],
                 '변환': report['transform'], 'lambda': report['transform_lambda'],
                 '변환 처리': report['transform_fit']['used'], '변환 상태': report['transform_fit']['status'],
                 '변환 참고': report['transform_fit']['note'],
                 '최대 동률 비율': report['transform_fit'].get('max_tie_fraction', np.nan),
                 '분석값 평균': report['mu_hat'], '분석값 표준편차': report['sigma_hat'],
                 '깊이별 후보': str(report['depth_count'][1:]),
                 '표시 타점 수': len(points), '경계 밖': int(points['outside'].fillna(False).sum()),
                 'raw 경계 밖': int(raw_out.sum()), '양쪽 경계 밖': int((raw_out & active_out).sum()),
                 'raw만 경계 밖': int((raw_out & ~active_out).sum()),
                 '분석값만 경계 밖': int((~raw_out & active_out).sum()),
                 '경계 없음': int(points['outside'].isna().sum()), '계산 초': report['seconds']})
    print(f"  완료: raw 평균 {report['mu_hat_raw']:.10g} · 분석 평균 {report['mu_hat']:.10g} · 깊이별 후보 {report['depth_count'][1:]} · 표시 {len(points):,}개 · {report['seconds']:.2f}초", flush=True)
part_summary = pd.DataFrame(rows).set_index('part_id')
display(part_summary)


## 3. 변환 전·후 웨이퍼 y 분포
Part별 **고유 웨이퍼**의 raw y와 분석값을 비교합니다. 위 행은 히스토그램과 같은 평균·표준편차의 정규 밀도, 아래 행은 정규 Q–Q plot입니다. OFF이면 두 열 모두 raw y입니다.

히스토그램은 극단값을 자르지 않고 전체 범위를 표시합니다. Q–Q plot은 양쪽 꼬리를 포함하며, 웨이퍼가 5,000장을 넘으면 표시 점만 순위별로 균등하게 줄입니다. 순위 계산·λ 추정과 모든 통계는 전체 웨이퍼를 사용합니다. 정규 Q–Q 기준선은 전체 평균·표준편차로 그립니다. 정규성이 좋아졌는지, 꼬리의 큰 값이 얼마나 압축됐는지 함께 확인하세요.

`DISTRIBUTION_PART_IDS = None`이면 모든 part, 예를 들어 `['PRODUCT_A']`이면 해당 part만 표시합니다. 각 단위에서 정확히 0인 값의 비율과 최대 동률 비율도 표에서 확인할 수 있습니다. Raw의 0은 순위변환 후 일반적으로 0이 아닌 동일 점수로 바뀝니다.


In [ ]:
DISTRIBUTION_PART_IDS = None  # None: 모든 part. 예: ['PRODUCT_A']


def plot_y_distributions(report):
    from statistics import NormalDist
    wf = report['wafer_values']
    raw_y = wf['raw_y'].to_numpy(dtype=float)
    analysis_y = wf['analysis_y'].to_numpy(dtype=float)
    enabled = report['transform'] != 'raw'
    after_label = 'After: ' + analysis_value_label(report)
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
    n = len(raw_y)
    ranks = np.unique(np.linspace(0, n - 1, min(n, 5000), dtype=int))
    normal = NormalDist()
    theoretical = np.array([normal.inv_cdf((int(i) + 0.5) / n) for i in ranks])
    diagnostics = []
    for col, (label, values) in enumerate([('Before: raw y', raw_y), (after_label, analysis_y)]):
        mu = float(np.mean(values))
        sd = float(np.std(values, ddof=1))
        vmin, vmax = float(values.min()), float(values.max())
        # 저카운트 정수는 정수마다 한 bin, 그 외에는 유한한 수의 균등 bin을 사용한다.
        if vmax - vmin <= 60 and np.equal(values, np.round(values)).all():
            bins = np.arange(vmin - 0.5, vmax + 1.5, 1.0)
        else:
            bins = min(60, max(10, int(np.sqrt(n))))
        ax = axes[0, col]
        ax.hist(values, bins=bins, density=True, alpha=0.6, color='#3878ad', label='Wafer values')
        xs = np.linspace(vmin, vmax, 400)
        density = np.exp(-0.5 * ((xs - mu) / sd)**2) / (sd * np.sqrt(2 * np.pi))
        ax.plot(xs, density, color='#d16a35', linewidth=1.5, label='Normal: same mean / SD')
        ax.set(title=label, xlabel='y (raw)' if col == 0 or not enabled else 'T(y)', ylabel='Density')
        ax.legend(fontsize=8)
        qq = axes[1, col]
        ordered = np.sort(values)
        qq.scatter(theoretical, ordered[ranks], s=8, alpha=0.45, color='#3878ad')
        qq.plot(theoretical, mu + sd * theoretical, color='#d16a35', linewidth=1.5)
        qq.set(title='Normal Q–Q plot', xlabel='Theoretical standard-normal quantile',
               ylabel='Observed raw y' if col == 0 or not enabled else 'Observed T(y)')
        for a in (ax, qq):
            a.grid(alpha=0.2)
            a.ticklabel_format(axis='both', style='plain', useOffset=False)
        diagnostics.append({'단위': label, '웨이퍼 수': n, '평균': mu, '표준편차': sd,
                            '왜도': float(pd.Series(values).skew()),
                            '0 비율': float(np.mean(values == 0)),
                            '최대 동률 비율': float(np.unique(values, return_counts=True)[1].max() / n),
                            '최솟값': vmin, '최댓값': vmax})
    mode = analysis_mode_label(report)
    fig.suptitle(f"part_id = {report['part_id']}  |  {mode}  |  K={n:,}")
    plt.show()
    display(pd.DataFrame(diagnostics).set_index('단위'))
    return fig


distribution_parts = (list(results_by_part) if DISTRIBUTION_PART_IDS is None
                      else ([str(DISTRIBUTION_PART_IDS)] if isinstance(DISTRIBUTION_PART_IDS, (str, int))
                            else list(map(str, DISTRIBUTION_PART_IDS))))
for part_id in distribution_parts:
    if part_id not in results_by_part:
        print(f'part {part_id}: 완료된 분석 결과가 없습니다.')
        continue
    part_heading(part_id)
    plot_y_distributions(results_by_part[part_id])


## 4. part별 차트와 폭의 원인
위에는 선택한 단위의 조합 평균과 envelope, 가운데에는 타점 수 $m_n$, 아래에는 표준오차 $\hat\sigma_p/\sqrt n$을 표시합니다.
선택에 따라 순위 정규점수 / Yeo–Johnson / raw 단위입니다. 제목에 모드를 표시하고, λ는 Yeo–Johnson에만 표시합니다.
분석 단위의 웨이퍼 표준편차 $\hat\sigma_p$는 part마다 하나입니다. 표준오차는 $1/\sqrt n$으로 감소하고, 경계의 불규칙한 변화는 $m_n$에서 옵니다.


In [ ]:
def plot_part(report):
    points, summary = report['points'], report['envelope']
    part_id, mu_hat = report['part_id'], report['mu_hat']
    value_label = analysis_value_label(report)
    mode_label = analysis_mode_label(report)
    if points.empty:
        print(f'part {part_id}: MIN_N / MAX_DEPTH / STEP 조건에 맞는 관측 조합이 없습니다.')
        return None
    fig, (ax, axm, axs) = plt.subplots(3, 1, figsize=(13, 8.5), sharex=True,
                                 gridspec_kw={'height_ratios': [4, 1, 1], 'hspace': 0.08})
    inside = points['outside'].eq(False).fillna(False).to_numpy(bool)
    outside = points['outside'].fillna(False).to_numpy(bool)
    ax.scatter(points.loc[inside, 'n'], points.loc[inside, 'sample_mean'], s=10,
               alpha=0.25, color='#65758b', label=f'경계 안 {inside.sum():,}개')
    ax.scatter(points.loc[outside, 'n'], points.loc[outside, 'sample_mean'], s=24,
               alpha=0.9, color='#d44838', label=f'경계 밖 {outside.sum():,}개', zorder=4)
    x = summary['n'].to_numpy(float)
    lower, upper = summary['lower'].to_numpy(float), summary['upper'].to_numpy(float)
    ax.fill_between(x, lower, upper, color='#2789d2', alpha=0.10)
    ax.plot(x, upper, '.-', color='#2789d2', linewidth=1.3, markersize=3,
            label=r'$\hat{\mu}_p \pm \hat{\sigma}_p\sqrt{2\ln m_{p,n}/n}$')
    ax.plot(x, lower, '.-', color='#2789d2', linewidth=1.3, markersize=3)
    ax.axhline(mu_hat, color='#202b3b', linestyle='--', linewidth=1.3,
               label=f'part 평균 = {mu_hat:.5g}')
    ax.set_title(f'part_id = {part_id}  |  {mode_label} · 후보 {len(points):,}개', loc='left', pad=14)
    ax.set_ylabel(f'조합의 {value_label} 평균')
    ax.ticklabel_format(axis='y', style='plain', useOffset=False)
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), frameon=False)
    axm.vlines(x, 0, summary['m_n'].to_numpy(), color='#344964', linewidth=1)
    axm.scatter(x, summary['m_n'], color='#344964', s=6)
    axm.set_ylabel('타점 수 m_n')
    axs.plot(x, summary['sigma_hat_n'], '.-', color='#8e63aa', linewidth=1)
    axs.set(xlabel='조합의 고유 웨이퍼 수 n', ylabel=r'$\hat{\sigma}_p/\sqrt{n}$')
    axm.set_ylim(bottom=0)
    if LOG_X:
        axm.set_xscale('log')
        axm.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value:g}'))
    for a in (ax, axm, axs):
        a.grid(alpha=0.2)
        a.spines[['top', 'right']].set_visible(False)
    fig.subplots_adjust(right=0.72, left=0.10, top=0.90, bottom=0.09)
    plt.show()
    return fig

for part_id, report in results_by_part.items():
    part_heading(part_id)
    plot_part(report)

## 5. n별 계산 값과 경계 밖 조합
`sample_mean`은 선택한 분석 단위의 조합 평균, `raw_mean`은 동일한 조합의 raw 산술평균입니다.
`mu_hat`, `sigma_hat`은 해당 part의 고유 웨이퍼 **분석값** 평균과 표본표준편차입니다.
`sigma_hat_n = sigma_hat / sqrt(n)`, `half_width = sigma_hat_n × sqrt_2log_m`으로 경계를 계산합니다.
`mu_hat_raw`, `sigma_hat_raw`는 원단위 참고값이며 ON 경계에는 사용하지 않습니다.
`transform`, `transform_fit_method`, `transform_fit_status`, `transform_fit_note`로 변환 방식과 주의사항을 확인합니다. Yeo–Johnson의 λ와 추정법은 `transform_lambda`, `lambda_method_used`에 저장됩니다. 순위변환에는 λ가 없으므로 빈 값입니다. 동률 정보는 `rank_n_unique`, `rank_max_tie_fraction`에 저장됩니다. 비교표는 raw 평균 차이의 절댓값이 큰 순서로 표시합니다.

아래 비교표는 **동일한 공정조합**에 대한 raw 경계와 분석 경계의 판정을 함께 표시합니다.
`raw만 경계 밖`인 조합은 변환 후 경계 안으로 들어온 조합입니다. 방향은 `raw_direction`과 `direction`을 확인하세요.
raw 경계는 변환 전후 비교 기준이며, 실제 이상 여부의 정답은 아닙니다. 표시 행 수는 `OUTSIDE_ROWS`로 제한되지만 전체 결과는 `report['points']`에 있습니다.


In [ ]:
for part_id, report in results_by_part.items():
    part_heading(part_id)
    fit = report['transform_fit']
    print(f"변환: {analysis_mode_label(report)} / 상태={fit['status']} / 웨이퍼={fit['n_fit']:,}")
    if report['transform'] == 'yeo-johnson':
        print(f"  λ 적합 가중치 양수={fit['n_used_last']:,}/{fit['n_fit']:,}")
    if fit['note']:
        print(f"  사유: {fit['note']}")
    display(report['envelope'])
    points = report['points']
    counts = points['detection_comparison'].value_counts().reindex(
        ['양쪽 경계 밖', 'raw만 경계 밖', '분석값만 경계 밖', '양쪽 경계 안'], fill_value=0)
    display(counts.rename('조합 수').to_frame())
    # 변환 후 경계 안으로 들어온 raw 후보도 검토 표에서 사라지지 않게 한다.
    review = points.loc[points['outside'].fillna(False) | points['raw_outside'].fillna(False)].copy()
    if review.empty:
        print('두 기준 모두 경계 밖 조합이 없습니다.')
    else:
        review = review.loc[review['raw_mean_difference'].abs().sort_values(ascending=False).index]
        display(review[['step', 'depth', 'path', 'n', 'raw_mean', 'raw_mean_difference',
                        'raw_lower', 'raw_upper', 'raw_direction', 'sample_mean', 'mu_hat',
                        'm_n', 'sigma_hat', 'sigma_hat_n', 'lower', 'upper', 'direction',
                        'detection_comparison', 'transform', 'transform_lambda']].head(OUTSIDE_ROWS))


## 선택: 결과 저장 / 제품 하나 다시 보기
설정의 `SAVE_DIR`를 지정하면 제품별 평균 타점과 envelope 계산표를 CSV로 저장합니다. raw 평균과 분석 단위·λ·실제 추정 방식 및 전환 사유도 함께 저장합니다.
아래 `PART_TO_VIEW`를 지정하면 이미 계산한 해당 제품의 차트만 다시 표시합니다.

In [ ]:
if SAVE_DIR is not None:
    save_dir = Path(SAVE_DIR).expanduser()
    save_dir.mkdir(parents=True, exist_ok=True)
    part_summary.to_csv(save_dir / 'part_summary.csv', encoding='utf-8-sig')
    for part_id, report in results_by_part.items():
        safe = re.sub(r'[^\w.-]+', '_', str(part_id))[:80] or 'part'
        suffix = hashlib.sha256(str(part_id).encode()).hexdigest()[:8]
        stem = f'{safe}_{suffix}'
        report['points'].to_csv(save_dir / f'{stem}_means.csv', index=False, encoding='utf-8-sig')
        report['envelope'].to_csv(save_dir / f'{stem}_envelope.csv', index=False, encoding='utf-8-sig')
        report['raw_envelope'].to_csv(save_dir / f'{stem}_raw_envelope.csv', index=False, encoding='utf-8-sig')
    print(f'저장 위치: {save_dir.resolve()}')

PART_TO_VIEW = None  # 예: 'PRODUCT_A'
if PART_TO_VIEW is not None:
    plot_part(results_by_part[str(PART_TO_VIEW)])